# SDESK Issue Export

Fetches all issues from the **Granton AI – Customer Service Management** (SDESK) Jira project and exports them to an Excel file that is then uploaded to Google Drive via the internal Drive integration API.

## Sections
1. **Setup** — packages, config
2. **Fetch** — paginated Jira REST API pull
3. **Data Processing** — parse & enrich into DataFrame
4. **Summary** — counts by status, type, priority
5. **Volume Over Time** — monthly trend
6. **Status & Priority Distribution** — charts
7. **Assignee Workload** — open tickets per person
8. **Reporter / Client View** — who raises most tickets
9. **Open Issues Aging** — how long issues sit unresolved
10. **Resolution Time** — time-to-close for resolved issues
11. **Export to Excel** — multi-sheet workbook
12. **Upload to Google Drive** — push the file via the Drive integration API

## 1. Setup

In [ ]:
# Uncomment on first run
# !pip install requests pandas matplotlib seaborn openpyxl

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import date, datetime
import os

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

# ── SECRETS ──────────────────────────────────────────────────
# Loaded from the .env file in THIS project folder (git-ignored).
# First run:  cp .env.example .env  and fill it in.
# Start Jupyter from this folder so the kernel's working directory matches.
import os
from pathlib import Path

from dotenv import load_dotenv

ENV_PATH = Path.cwd() / '.env'
if not ENV_PATH.is_file():
    raise RuntimeError(
        f'No .env found at {ENV_PATH}. Copy .env.example to .env in this '
        f'project folder and fill it in (and start Jupyter from this folder).'
    )
load_dotenv(ENV_PATH)


def env(name, default=None, required=True):
    """Read a setting from this folder's .env / the environment."""
    val = os.getenv(name, default)
    if required and not val:
        raise RuntimeError(
            f'Missing required setting: {name}. Add it to {ENV_PATH} '
            f'(see .env.example).'
        )
    return val

# ── JIRA CONFIG ───────────────────────────────────────────────
JIRA_URL     = f"https://{env('JIRA_DOMAIN')}"
JIRA_EMAIL   = env('JIRA_EMAIL')
JIRA_TOKEN   = env('JIRA_TOKEN')
JIRA_PROJECT = env('JIRA_PROJECT', 'SDESK')

# ── REPORT PERIOD ─────────────────────────────────────────────
REPORT_YEAR  = date.today().year
REPORT_MONTH = date.today().month

# Optional extra JQL filter, e.g. 'AND created >= "2026-01-01"'
JQL_EXTRA = ''

# ── DRIVE INTEGRATION API ─────────────────────────────────────
BASE_URL         = env('GDRIVE_API_BASE_URL')   # Drive integration API base URL (currently UAT)
GDRIVE_FOLDER_ID = env('GDRIVE_FOLDER_ID')     # last segment of the target Drive folder URL
CLIENT_ID        = env('GDRIVE_CLIENT_ID')     # UUID identifying this client in the Drive API

# ── OUTPUT ────────────────────────────────────────────────────
EXCEL_PATH = datetime.now().strftime('sdesk_export_%Y_%m_%d_%H_%M.xlsx')

print(f'Report period : {REPORT_YEAR}-{REPORT_MONTH:02d}')
print(f'Jira project  : {JIRA_PROJECT}')
print(f'Output file   : {EXCEL_PATH}')
print(f'Drive API     : {BASE_URL}')

## 2. Fetch Issues from Jira

In [ ]:
def jira_get(endpoint, params=None):
    resp = requests.get(
        f'{JIRA_URL}/rest/api/3/{endpoint}',
        auth=(JIRA_EMAIL, JIRA_TOKEN),
        headers={'Accept': 'application/json'},
        params=params,
    )
    resp.raise_for_status()
    return resp.json()


def jira_post(endpoint, body):
    resp = requests.post(
        f'{JIRA_URL}/rest/api/3/{endpoint}',
        auth=(JIRA_EMAIL, JIRA_TOKEN),
        headers={'Accept': 'application/json', 'Content-Type': 'application/json'},
        json=body,
    )
    resp.raise_for_status()
    return resp.json()


# Discover SLA custom fields (Jira Service Management stores them as sd-sla-field)
_all_fields = jira_get('field')
SLA_FIELDS = {
    f['id']: f['name']
    for f in _all_fields
    if f.get('schema', {}).get('custom') == 'com.atlassian.servicedesk:sd-sla-field'
}
print('SLA fields found:', SLA_FIELDS)

STANDARD_FIELDS = [
    'summary', 'status', 'issuetype', 'priority',
    'created', 'updated', 'resolutiondate',
    'assignee', 'reporter', 'labels', 'components',
    'resolution', 'comment',
]

JQL = f'project = {JIRA_PROJECT} {JQL_EXTRA} ORDER BY created DESC'


def _fetch_page_by_page(jql, fields, page_size=100):
    issues, next_page_token = [], None
    data = {}
    while True:
        body = {'jql': jql, 'maxResults': page_size, 'fields': fields}
        if next_page_token:
            body['nextPageToken'] = next_page_token
        data = jira_post('search/jql', body)
        if data.get('errorMessages') or data.get('errors'):
            raise RuntimeError(data.get('errorMessages') or data.get('errors'))
        batch = data.get('issues', [])
        issues.extend(batch)
        next_page_token = data.get('nextPageToken')
        if not batch or not next_page_token:
            break
    total = data.get('total', len(issues))
    return issues, total


# Try with SLA field IDs; if the search comes back empty, fall back to standard fields only.
# (Some Jira configurations reject unknown/restricted field IDs silently and return 0 results.)
FETCH_FIELDS = STANDARD_FIELDS + list(SLA_FIELDS.keys())
raw_issues, _total = _fetch_page_by_page(JQL, FETCH_FIELDS)

if not raw_issues and SLA_FIELDS:
    print('Warning: 0 issues with SLA fields — retrying without them.')
    FETCH_FIELDS = STANDARD_FIELDS
    SLA_FIELDS   = {}          # no SLA data available via search
    raw_issues, _total = _fetch_page_by_page(JQL, FETCH_FIELDS)

print(f'Fetched {len(raw_issues)} / {_total} issues  |  SLA fields in response: {list(SLA_FIELDS.values())}')

## 3. Data Processing

In [ ]:
def _str(d, *keys, default=''):
    val = d
    for k in keys:
        if not isinstance(val, dict):
            return default
        val = val.get(k)
    return val if val is not None else default


def adf_to_text(node):
    """Recursively extract plain text from an Atlassian Document Format node."""
    if not isinstance(node, dict):
        return ''
    if node.get('type') == 'text':
        return node.get('text', '')
    block_types = {
        'paragraph', 'heading', 'bulletList', 'orderedList',
        'listItem', 'codeBlock', 'blockquote', 'panel', 'rule',
    }
    children = [adf_to_text(c) for c in node.get('content', [])]
    sep = '\n' if node.get('type') in block_types else ''
    return sep.join(c for c in children if c)


def parse_sla(sla_data):
    """Return (breached, remaining_friendly, elapsed_friendly) from a JSM SLA field value."""
    if not sla_data:
        return None, '', ''
    ongoing   = sla_data.get('ongoingCycle') or {}
    completed = sla_data.get('completedCycles') or []
    cycle     = ongoing if ongoing else (completed[-1] if completed else {})
    if not cycle:
        return None, '', ''
    breached  = cycle.get('breached', False)
    remaining = (ongoing.get('remainingTime') or {}).get('friendly', '')
    elapsed   = (cycle.get('elapsedTime') or {}).get('friendly', '')
    return breached, remaining, elapsed


def parse_issues(issues):
    rows = []
    for i in issues:
        f = i['fields']

        # Last comment (ADF → plain text)
        comments = (f.get('comment') or {}).get('comments', [])
        if comments:
            last             = comments[-1]
            last_comment      = adf_to_text(last.get('body') or {})
            last_comment_user = _str(last, 'author', 'displayName')
        else:
            last_comment      = ''
            last_comment_user = ''

        row = {
            'key':              i['key'],
            'url':              f'{JIRA_URL}/browse/{i["key"]}',
            'summary':          f.get('summary', ''),
            'issuetype':        _str(f, 'issuetype', 'name'),
            'status':           _str(f, 'status', 'name'),
            'status_category':  _str(f, 'status', 'statusCategory', 'name'),
            'priority':         _str(f, 'priority', 'name'),
            'assignee':         _str(f, 'assignee', 'displayName') or 'Unassigned',
            'reporter':         _str(f, 'reporter', 'displayName'),
            'reporter_email':   _str(f, 'reporter', 'emailAddress'),
            'resolution':       _str(f, 'resolution', 'name'),
            'labels':           ', '.join(f.get('labels') or []),
            'components':       ', '.join(c.get('name', '') for c in (f.get('components') or [])),
            'comment_count':    _str(f, 'comment', 'total') or 0,
            'last_comment':     last_comment,
            'last_comment_user': last_comment_user,
            'created':          f.get('created'),
            'updated':          f.get('updated'),
            'resolutiondate':   f.get('resolutiondate'),
        }

        # SLA fields — one set of columns per discovered SLA
        for fid, fname in SLA_FIELDS.items():
            breached, remaining, elapsed = parse_sla(f.get(fid))
            row[f'{fname} — breached']   = breached
            row[f'{fname} — remaining']  = remaining
            row[f'{fname} — elapsed']    = elapsed

        rows.append(row)

    df = pd.DataFrame(rows)

    for col in ('created', 'updated', 'resolutiondate'):
        df[col] = pd.to_datetime(df[col], utc=True).dt.tz_convert('Europe/Prague').dt.tz_localize(None)

    now = pd.Timestamp.now()
    df['age_days']        = (now - df['created']).dt.days
    df['resolution_days'] = (df['resolutiondate'] - df['created']).dt.days
    df['month']           = df['created'].dt.to_period('M').astype(str)
    df['week']            = df['created'].dt.to_period('W').astype(str)
    df['is_open']         = df['status_category'].isin(['To Do', 'In Progress'])

    return df


df = parse_issues(raw_issues)
print(f'Shape: {df.shape}')
df.head()

## 4. Summary

In [ ]:
total       = len(df)
open_count  = df['is_open'].sum()
closed_count = (~df['is_open']).sum()
avg_age_open = df[df['is_open']]['age_days'].mean()
avg_res_days = df['resolution_days'].dropna().mean()

print(f'Total issues      : {total}')
print(f'Open              : {open_count}')
print(f'Closed/Done       : {closed_count}')
print(f'Avg age (open)    : {avg_age_open:.1f} days')
print(f'Avg resolution    : {avg_res_days:.1f} days')

# ── Summary tables ────────────────────────────────────────────
df_by_status   = df.groupby('status').size().reset_index(name='count').sort_values('count', ascending=False)
df_by_type     = df.groupby('issuetype').size().reset_index(name='count').sort_values('count', ascending=False)
df_by_priority = df.groupby('priority').size().reset_index(name='count').sort_values('count', ascending=False)

print('\n--- By Status ---')
print(df_by_status.to_string(index=False))
print('\n--- By Type ---')
print(df_by_type.to_string(index=False))
print('\n--- By Priority ---')
print(df_by_priority.to_string(index=False))

## 5. Volume Over Time

In [ ]:
monthly = df.groupby('month').size().reset_index(name='issues')
monthly['cumulative'] = monthly['issues'].cumsum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(monthly['month'], monthly['issues'], alpha=0.7, label='Monthly')
ax2 = ax.twinx()
ax2.plot(monthly['month'], monthly['cumulative'], color='tab:red', marker='o', label='Cumulative')
ax.set_title(f'SDESK Issue Volume Over Time')
ax.set_ylabel('Issues / month')
ax2.set_ylabel('Cumulative')
ax.tick_params(axis='x', rotation=45)
l1, lb1 = ax.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1+l2, lb1+lb2, loc='upper left')
plt.tight_layout()
plt.show()
monthly

## 6. Status & Priority Distribution

In [ ]:
PRIORITY_ORDER = ['Highest', 'High', 'Medium', 'Low', 'Lowest']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Status
axes[0].barh(df_by_status['status'], df_by_status['count'])
axes[0].bar_label(axes[0].containers[0], padding=3)
axes[0].set_title('Issues by Status')
axes[0].set_xlabel('Count')

# Priority (sorted)
pri_ordered = df_by_priority.set_index('priority').reindex(
    [p for p in PRIORITY_ORDER if p in df_by_priority['priority'].values]
).dropna().reset_index()
colors = ['#d62728', '#ff7f0e', '#1f77b4', '#2ca02c', '#9467bd']
axes[1].barh(pri_ordered['priority'], pri_ordered['count'], color=colors[:len(pri_ordered)])
axes[1].bar_label(axes[1].containers[0], padding=3)
axes[1].set_title('Issues by Priority')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 7. Issue Type Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(df_by_type['issuetype'], df_by_type['count'])
ax.bar_label(ax.containers[0], padding=3)
ax.set_title('Issues by Type')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

## 8. Assignee Workload

In [ ]:
df_assignee = (
    df.groupby(['assignee', 'is_open'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={True: 'open', False: 'closed'})
    .assign(total=lambda x: x.get('open', 0) + x.get('closed', 0))
    .sort_values('total', ascending=True)
)

fig, ax = plt.subplots(figsize=(12, max(4, len(df_assignee) * 0.5)))
df_assignee[['closed', 'open']].plot(kind='barh', stacked=True, ax=ax, color=['#2ca02c', '#ff7f0e'])
ax.set_title('Issues per Assignee (open vs closed)')
ax.set_xlabel('Count')
ax.set_ylabel('')
ax.legend(['Closed', 'Open'])
plt.tight_layout()
plt.show()
df_assignee

## 9. Reporter / Client View

In [ ]:
df_reporter = (
    df.groupby('reporter')
    .agg(
        total=('key', 'count'),
        open=('is_open', 'sum'),
        email=('reporter_email', 'first'),
    )
    .sort_values('total', ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(12, max(5, len(df_reporter) * 0.45)))
ax.barh(df_reporter.index[::-1], df_reporter['total'][::-1], alpha=0.8)
ax.set_title('Top 20 Reporters (clients) by Issue Count')
ax.set_xlabel('Issues')
ax.set_ylabel('')
plt.tight_layout()
plt.show()
df_reporter

## 10. Open Issues Aging

In [ ]:
BUCKETS = [0, 1, 3, 7, 14, 30, 60, 90, float('inf')]
LABELS  = ['0d', '1-2d', '3-6d', '7-13d', '14-29d', '30-59d', '60-89d', '90d+']

df_open = df[df['is_open']].copy()
df_open['age_bucket'] = pd.cut(df_open['age_days'], bins=BUCKETS, labels=LABELS, right=False)
aging = df_open.groupby('age_bucket', observed=True).size().reset_index(name='count')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ca02c', '#98df8a', '#ffbb78', '#ff7f0e', '#d62728', '#8c1a1a', '#5c1111', '#3c0000']
axes[0].bar(aging['age_bucket'].astype(str), aging['count'], color=colors[:len(aging)])
axes[0].set_title(f'Open Issues by Age ({int(df_open["age_days"].max())} max days)')
axes[0].set_xlabel('Age bucket')
axes[0].set_ylabel('Issues')

# Aging by assignee (open only)
aging_assignee = (
    df_open.groupby(['assignee', 'age_bucket'], observed=True)
    .size()
    .unstack(fill_value=0)
)
aging_assignee.plot(kind='barh', stacked=True, ax=axes[1], colormap='RdYlGn_r')
axes[1].set_title('Open Issues Aging by Assignee')
axes[1].set_xlabel('Issues')
axes[1].set_ylabel('')
axes[1].legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.show()
df_open[['key', 'summary', 'status', 'priority', 'assignee', 'age_days', 'created']].sort_values('age_days', ascending=False).head(20)

## 11. Resolution Time

In [ ]:
df_resolved = df.dropna(subset=['resolution_days']).copy()

if df_resolved.empty:
    print('No resolved issues with resolution dates yet.')
else:
    print(df_resolved['resolution_days'].describe().round(1))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(df_resolved['resolution_days'].clip(upper=60), bins=30, color='steelblue', edgecolor='white')
    axes[0].set_title('Resolution Time Distribution (capped at 60 days)')
    axes[0].set_xlabel('Days to resolve')
    axes[0].set_ylabel('Issues')

    rt_by_type = (
        df_resolved.groupby('issuetype')['resolution_days']
        .median()
        .sort_values(ascending=True)
    )
    rt_by_type.plot(kind='barh', ax=axes[1], color='steelblue')
    axes[1].set_title('Median Resolution Time by Issue Type (days)')
    axes[1].set_xlabel('Days')
    axes[1].set_ylabel('')

    plt.tight_layout()
    plt.show()

## 12. Monthly Heatmap — Issues by Status Category

In [ ]:
heat = (
    df.groupby(['month', 'status_category'])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(14, max(3, len(heat.columns) * 0.8)))
sns.heatmap(
    heat.T, annot=True, fmt='d', cmap='Blues',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Issues'}
)
ax.set_title('Issues per Month by Status Category')
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 13. Export to Excel

In [ ]:
sla_cols = [
    f'{name} — {metric}'
    for name in SLA_FIELDS.values()
    for metric in ('breached', 'remaining', 'elapsed')
]

EXPORT_COLS = [
    'key', 'summary', 'issuetype', 'status',
    'priority', 'assignee', 'reporter', 'reporter_email',
    'labels',
    'comment_count', 'last_comment', 'last_comment_user',
    'created', 'updated', 'resolutiondate',
    'age_days',
    *sla_cols,
    'url',
]

df_export = df[EXPORT_COLS].sort_values('created', ascending=False).copy()

for col in ('created', 'updated', 'resolutiondate'):
    df_export[col] = df_export[col].dt.strftime('%Y-%m-%d %H:%M')

with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:

    # Sheet 1 — All issues
    df_export.to_excel(writer, sheet_name='All Issues', index=False)

    # Sheet 2 — Open issues only
    df_export[df['is_open'].reindex(df_export.index)].to_excel(writer, sheet_name='Open Issues', index=False)

    # Sheet 3 — Summary
    summary_rows = [
        ['Metric', 'Value'],
        ['Total issues', total],
        ['Open', int(open_count)],
        ['Closed / Done', int(closed_count)],
        ['Avg age open (days)', round(avg_age_open, 1)],
        ['Avg resolution (days)', round(avg_res_days, 1) if not pd.isna(avg_res_days) else 'N/A'],
    ]
    pd.DataFrame(summary_rows[1:], columns=summary_rows[0]).to_excel(
        writer, sheet_name='Summary', index=False
    )

    # Sheet 4 — By status
    df_by_status.to_excel(writer, sheet_name='By Status', index=False)

    # Sheet 5 — By assignee
    df_assignee.reset_index().to_excel(writer, sheet_name='By Assignee', index=False)

    # Sheet 6 — By reporter
    df_reporter.reset_index().to_excel(writer, sheet_name='By Reporter', index=False)

    # Sheet 7 — Monthly volume
    monthly.to_excel(writer, sheet_name='Monthly Volume', index=False)

print(f'Saved → {EXCEL_PATH}')
print(f'Export columns: {EXPORT_COLS}')

## 14. Upload to Google Drive

Uploads the Excel file via the internal **Drive integration API** (`POST /upload-file`).

- Set `BASE_URL`, `GDRIVE_FOLDER_ID`, and `CLIENT_ID` in the config cell (section 1).
- `exists=True` tells the API to overwrite an existing file with the same name instead of creating a duplicate.

In [ ]:
def upload_to_drive(local_path, folder_id, client_id, base_url=BASE_URL):
    if not os.path.exists(local_path):
        raise FileNotFoundError(f'File not found: {local_path} — re-run the export cell first.')

    with open(local_path, 'rb') as f:
        resp = requests.post(
            f'{base_url}/upload-file',
            data={
                'folder_id': folder_id,
                'client_id': client_id,
                'exists':    'false',   # always create new (filename is timestamped)
            },
            files={'file': (os.path.basename(local_path), f,
                            'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')},
        )
    if not resp.ok:
        print(f'HTTP {resp.status_code}  {resp.reason}')
        print(resp.text)
        resp.raise_for_status()
    result = resp.json()
    print(f'Uploaded → {os.path.basename(local_path)}')
    print(f'Info     : {result["info"]}')
    print(f'Link     : {result["file_weblink"]}')
    return result


upload_to_drive(EXCEL_PATH, GDRIVE_FOLDER_ID, CLIENT_ID)